# GEN-MAMA-MX — Pipeline Completo TCGA-BRCA

Pipeline automatizado de precision oncology para cáncer de mama.

**Módulos:**
- **M1** — Descarga de mutaciones somáticas TCGA-BRCA (GDC API)
- **M2** — Priorización de variantes missense por frecuencia
- **M3** — Generación de FASTA WT/MUT + predicción estructural Boltz-1
- **M4** — Docking molecular con AutoDock Vina
- **M5** — Exportación de resultados para la app web local

**Instrucciones:**
1. Sube este notebook a Google Colab
2. Activa GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU
3. Ejecuta la celda 0 PRIMERO y **reinicia el entorno cuando se indique**
4. Ejecuta el resto de celdas en orden
5. Al final descarga `gemamx_data.zip` y descomprímelo en `D:/ProyectosIA/GEMAMX/data/`

In [ ]:
# @title ⚙️ 0. Configuración del Entorno
# @markdown **IMPORTANTE:** Tras ejecutar esta celda, reinicia el entorno
# @markdown (Entorno de ejecución → Reiniciar entorno) y luego continúa desde la celda 0b.
import os, subprocess, sys

IS_COLAB = 'google.colab' in str(get_ipython())

if IS_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    BASE_PATH = '/content/drive/MyDrive/GEMAMX'
    for d in ['data/fasta', 'data/structures', 'data/docking/results',
               'data/docking/ligands', 'data/docking/receptors', 'scripts']:
        os.makedirs(f'{BASE_PATH}/{d}', exist_ok=True)

    # ── Fix 1: NumPy/pandas binary incompatibility ────────────────────────
    print('Paso 1/3 — Fijando NumPy y pandas...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        'numpy==1.26.4', 'pandas==2.2.2', 'biopython==1.83', 'requests', 'tqdm',
        '--quiet', '--force-reinstall'
    ], check=False)

    # ── Fix 2: PyTorch Lightning version compatible con Boltz ─────────────
    # Colab trae pytorch_lightning>=2.x pero Boltz requiere 1.x API.
    # El error 'LeafSpec is deprecated' y el crash silencioso de PIK3CA
    # son causados por esta incompatibilidad.
    print('Paso 2/3 — Fijando pytorch-lightning compatible con Boltz...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        'pytorch-lightning==2.1.4',   # ultima version 2.x estable que Boltz soporta
        'lightning==2.1.4',
        '--quiet', '--force-reinstall'
    ], check=False)

    # ── Boltz ─────────────────────────────────────────────────────────────
    print('Paso 3/3 — Instalando Boltz-1...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', 'boltz', '--quiet'
    ], check=False)

    # Symlinks para acceso directo
    for d in ['data', 'scripts']:
        link, target = f'/content/{d}', f'{BASE_PATH}/{d}'
        if os.path.islink(link): os.remove(link)
        if not os.path.exists(link): os.symlink(target, link)

    print()
    print('✅ Instalación completa.')
    print('⚠️  REINICIA EL ENTORNO AHORA:')
    print('   Entorno de ejecución → Reiniciar entorno')
    print('   Luego ejecuta desde la celda 0b hacia abajo.')
else:
    BASE_PATH = '.'
    print('Entorno local — BASE_PATH = .')

In [ ]:
# @title 0b. Restaurar accesos tras reinicio (ejecutar siempre después de reiniciar)
import os, sys
IS_COLAB = 'google.colab' in str(get_ipython())

if IS_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/GEMAMX'
    for d in ['data', 'scripts']:
        link, target = f'/content/{d}', f'{BASE_PATH}/{d}'
        os.makedirs(target, exist_ok=True)
        if os.path.islink(link): os.remove(link)
        if not os.path.exists(link): os.symlink(target, link)

    # Verificar versiones instaladas
    import numpy as np, pandas as pd
    print(f'✅ NumPy            {np.__version__}')
    print(f'✅ pandas           {pd.__version__}')
    try:
        import pytorch_lightning as pl
        print(f'✅ pytorch_lightning {pl.__version__}')
    except ImportError:
        print('⚠  pytorch_lightning no disponible — re-ejecuta celda 0')
    print(f'✅ Drive OK — {BASE_PATH}')
else:
    BASE_PATH = '.'
    import numpy as np, pandas as pd
    print(f'Local | NumPy {np.__version__} | pandas {pd.__version__}')

## M1 — Descarga de datos TCGA-BRCA desde GDC API

In [ ]:
# @title 1. Descarga de mutaciones somáticas (SSMs) desde GDC
import requests, json

PROJECT_ID  = 'TCGA-BRCA'
TOTAL_FETCH = 5000

def fetch_ssms(project_id, size=5000):
    filters = {'op': 'and', 'content': [
        {'op': 'in', 'content': {'field': 'occurrence.case.project.project_id', 'value': [project_id]}},
        {'op': 'in', 'content': {'field': 'consequence.transcript.consequence_type', 'value': ['missense_variant']}}
    ]}
    params = {
        'filters': json.dumps(filters),
        'fields': 'genomic_dna_change,consequence.transcript.gene.symbol,consequence.transcript.aa_change',
        'format': 'JSON', 'size': size
    }
    print(f'Descargando {size} SSMs de {project_id}...')
    r = requests.get('https://api.gdc.cancer.gov/ssms', params=params, timeout=60)
    r.raise_for_status()
    return r.json()

raw_data = fetch_ssms(PROJECT_ID, size=TOTAL_FETCH)
print(f'✓ Registros descargados: {len(raw_data["data"]["hits"]):,}')

## M2 — Procesamiento y Priorización de Variantes

In [ ]:
# @title 2. Parseo y construcción del DataFrame
import pandas as pd, re

rows = []
for hit in raw_data['data']['hits']:
    try:
        cons = hit['consequence'][0]['transcript']
        gene = cons['gene']['symbol']
        aa   = cons.get('aa_change', None)
        dna  = hit.get('genomic_dna_change', '')
        chrom, pos = '', 0
        m = re.match(r'chr([\w]+):g\.(\d+)', dna)
        if m: chrom, pos = m.group(1), int(m.group(2))
        if gene and aa and aa not in ('N/A', 'None', '') and aa.strip():
            rows.append({'hugo_symbol': gene, 'protein_change': aa,
                         'dna_change': dna, 'chromosome': chrom, 'start_position': pos})
    except (KeyError, IndexError):
        continue

df = pd.DataFrame(rows)
print(f'✓ Variantes válidas: {len(df):,} | Genes únicos: {df["hugo_symbol"].nunique():,}')
df.head(10)

In [ ]:
# @title 3. Priorización — Top variantes por frecuencia
import json, os

TOP_N = 20
PRIORITY_GENES = {
    'PIK3CA','TP53','ERBB2','BRCA1','BRCA2',
    'CDH1','GATA3','MAP3K1','KMT2C','PTEN',
    'RB1','ARID1A','SF3B1','AKT1','CBFB'
}
TOTAL_SAMPLES = max(df['dna_change'].nunique(), 1)

freq = (df.groupby(['hugo_symbol','protein_change','chromosome','start_position'])
          .size().reset_index(name='frequency'))
freq['priority'] = freq['hugo_symbol'].isin(PRIORITY_GENES).astype(int)
freq = freq.sort_values(['priority','frequency'], ascending=[False,False])
top  = freq.head(TOP_N).copy()
top['cohort_percentage'] = (top['frequency'] / TOTAL_SAMPLES * 100).round(2)

def classify(row):
    p, g = row['cohort_percentage'], row['hugo_symbol']
    if p >= 5.0 or (g in PRIORITY_GENES and p >= 1.0): return 'Patogénica'
    if p >= 1.0: return 'Probable Patogénica'
    if p >= 0.5: return 'VUS'
    return 'Baja Frecuencia'

top['classification'] = top.apply(classify, axis=1)
top['rank'] = range(1, len(top)+1)

records = top[['hugo_symbol','protein_change','chromosome','start_position',
               'frequency','cohort_percentage','classification','rank']].to_dict(orient='records')
for v in records:
    v['frequency'] = int(v['frequency'])
    v['start_position'] = int(v['start_position'])
    v['cohort_percentage'] = float(v['cohort_percentage'])
    v['rank'] = int(v['rank'])

os.makedirs('data', exist_ok=True)
with open('data/top_variants.json','w') as f:
    json.dump(records, f, indent=4, ensure_ascii=False)
print(f'✓ data/top_variants.json — {len(records)} variantes')
pd.DataFrame(records)[['rank','hugo_symbol','protein_change','frequency','cohort_percentage','classification']]

## M3a — Generación de secuencias FASTA WT y Mutante

In [ ]:
# @title 3.5 Generación de FASTA desde UniProt
import requests, re, io, time, json, os
from Bio import SeqIO
from Bio.Seq import MutableSeq

FASTA_DIR = 'data/fasta'
os.makedirs(FASTA_DIR, exist_ok=True)
_cache = {}

def get_uniprot_id(gene):
    if gene in _cache: return _cache[gene]
    r = requests.get('https://rest.uniprot.org/uniprotkb/search',
        params={'query': f'gene_exact:{gene} AND organism_id:9606 AND reviewed:true',
                'fields': 'accession', 'format': 'json', 'size': 1}, timeout=15)
    acc = r.json().get('results',[{}])[0].get('primaryAccession') if r.ok else None
    _cache[gene] = acc
    return acc

def apply_mutation(fasta_text, change):
    try:
        records = list(SeqIO.parse(io.StringIO(fasta_text), 'fasta'))
        if not records: return None, None
        seq = str(records[0].seq)
        m = re.search(r'([A-Z])(\d+)([A-Z])', change)
        if m:
            wt_aa, pos, mut_aa = m.group(1), int(m.group(2)), m.group(3)
            idx = pos - 1
            if 0 <= idx < len(seq) and seq[idx] == wt_aa:
                mut = MutableSeq(seq); mut[idx] = mut_aa
                return seq, str(mut)
            print(f'    ⚠ Pos {pos}: esperado "{wt_aa}", encontrado "{seq[idx] if idx < len(seq) else "OOB"}"')
    except Exception as e:
        print(f'    ✗ Error: {e}')
    return None, None

with open('data/top_variants.json') as f: variants = json.load(f)

ok_count = 0
for v in variants:
    gene, change = v['hugo_symbol'], v['protein_change']
    print(f"[{v['rank']:>2}] {gene} {change}")
    wt_path, mut_path = f'{FASTA_DIR}/{gene}_WT.fasta', f'{FASTA_DIR}/{gene}_{change}.fasta'
    if os.path.exists(wt_path) and os.path.exists(mut_path):
        print('    ↳ Ya existen'); ok_count += 1; continue
    acc = get_uniprot_id(gene)
    if not acc: print(f'    ✗ UniProt no encontrado'); continue
    r = requests.get(f'https://www.uniprot.org/uniprot/{acc}.fasta', timeout=15)
    if not r.ok: print(f'    ✗ FASTA no descargado'); continue
    wt_seq, mut_seq = apply_mutation(r.text, change)
    if not wt_seq: print('    ✗ Mutación no aplicable'); continue
    open(wt_path,  'w').write(f'>A|protein\n{wt_seq}\n')
    open(mut_path, 'w').write(f'>A|protein\n{mut_seq}\n')
    print(f'    ✓ WT ({len(wt_seq)} aa) | MUT ({len(mut_seq)} aa)')
    ok_count += 1; time.sleep(0.3)

print(f'\n✓ {ok_count}/{len(variants)} genes con FASTA disponibles')

## M3b — Predicción estructural con Boltz-1

> **Requiere GPU T4.** Tiempo estimado: 2–15 min por proteína.
> 
> **Límite automático:** proteínas >1400 aa se omiten (exceden RAM de Colab T4).

In [ ]:
# @title 4. Predicción estructural 3D con Boltz-1
import subprocess, glob, time, json, os, sys

STRUCTURES_DIR = 'data/structures'
os.makedirs(STRUCTURES_DIR, exist_ok=True)

MAX_AA_BOLTZ = 1400   # T4 Colab: ~12 GB RAM, Boltz necesita ~6-8 GB para 1400 aa

# Verificar GPU y versión de pytorch_lightning
try:
    import torch
    gpu = torch.cuda.is_available()
    print(f'GPU: {torch.cuda.get_device_name(0) if gpu else "NO DISPONIBLE"}')
    if gpu:
        total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'     VRAM: {total_gb:.1f} GB')
except ImportError:
    gpu = False

try:
    import pytorch_lightning as pl
    print(f'pytorch_lightning: {pl.__version__}')
    # Validar que es una versión compatible (2.1.x)
    major, minor = [int(x) for x in pl.__version__.split('.')[:2]]
    if major == 2 and minor >= 1:
        print('✅ Versión compatible con Boltz')
    else:
        print(f'⚠ Versión {pl.__version__} puede ser incompatible — re-ejecuta celda 0')
except ImportError:
    print('⚠ pytorch_lightning no disponible — re-ejecuta celda 0')

def read_fasta_seq(path):
    lines = open(path).read().strip().split('\n')
    return ''.join(l for l in lines if not l.startswith('>'))

with open('data/top_variants.json') as f:
    variants = json.load(f)

pred_log, skipped = [], []
# Deduplicar: el WT de cada gen solo se predice una vez
predicted_wt = set()

for v in variants:
    gene, change = v['hugo_symbol'], v['protein_change']

    for tag, fpath in [
        ('WT',  f'data/fasta/{gene}_WT.fasta'),
        ('MUT', f'data/fasta/{gene}_{change}.fasta')
    ]:
        # Evitar predecir el mismo WT varias veces cuando hay múltiples variantes del gen
        if tag == 'WT' and gene in predicted_wt:
            continue

        if not os.path.exists(fpath):
            print(f'  ⚠ FASTA no encontrado: {fpath} — omitiendo')
            continue

        name     = os.path.basename(fpath).replace('.fasta', '')
        expected = f'{STRUCTURES_DIR}/boltz_results_{name}/predictions/{name}/{name}_model_0.cif'

        if os.path.exists(expected):
            size_kb = os.path.getsize(expected) / 1024
            print(f'  ↳ {name}: ya existe ({size_kb:.0f} KB) — omitiendo')
            pred_log.append({'name': name, 'status': 'cached', 'path': expected})
            if tag == 'WT': predicted_wt.add(gene)
            continue

        # Verificar longitud
        seq = read_fasta_seq(fpath)
        if len(seq) > MAX_AA_BOLTZ:
            print(f'  ⚠ {name}: {len(seq)} aa > límite {MAX_AA_BOLTZ} aa — omitiendo')
            skipped.append({'name': name, 'length': len(seq)})
            if tag == 'WT': predicted_wt.add(gene)  # No repetir el aviso
            continue

        print(f'  Prediciendo {name} ({len(seq)} aa)...')
        t0  = time.time()
        cmd = ['boltz', 'predict', fpath, '--out_dir', STRUCTURES_DIR,
               '--use_msa_server', '--override']
        res = subprocess.run(cmd, capture_output=True, text=True)
        elapsed = time.time() - t0

        if os.path.exists(expected):
            size_kb = os.path.getsize(expected) / 1024
            print(f'    ✓ {name}_model_0.cif ({size_kb:.0f} KB) en {elapsed:.0f}s')
            pred_log.append({'name': name, 'status': 'ok', 'path': expected})
            if tag == 'WT': predicted_wt.add(gene)
        else:
            # Mostrar solo las últimas 3 líneas del STDERR para diagnóstico
            stderr_lines = [l for l in (res.stderr or '').split('\n') if l.strip()][-3:]
            print(f'    ✗ Falló en {elapsed:.0f}s')
            print(f'    STDERR: {" | ".join(stderr_lines)}')
            pred_log.append({'name': name, 'status': 'failed'})

# Resumen final
ok   = [x for x in pred_log if x['status'] in ('ok','cached')]
fail = [x for x in pred_log if x['status'] == 'failed']
print(f'\n── Resumen ─────────────────────────────')
print(f'✓ Listas:              {len(ok)}')
if fail:    print(f'✗ Fallidas:            {[x["name"] for x in fail]}')
if skipped: print(f'⚠ Omitidas (>1400 aa): {[x["name"]+" ("+str(x["length"])+" aa)" for x in skipped]}')
if fail:
    print()
    print('Si sigues viendo errores de pytorch_lightning:')
    print('  1. Detén la ejecución')
    print('  2. Re-ejecuta SOLO la celda 0 → reinicia → 0b → salta directo a celda 4')

## M4 — Docking Molecular con AutoDock Vina

> Requiere ligandos `.pdbqt` en `data/docking/ligands/`. Si no los tienes, omite esta sección.

In [ ]:
# @title 6. Docking con AutoDock Vina
import subprocess, glob, json, os, sys

!apt-get install -y autodock-vina openbabel --quiet 2>/dev/null
subprocess.run([sys.executable, '-m', 'pip', 'install', 'meeko', '--quiet'], check=False)

LIGANDS_DIR, RECEPTORS_DIR, RESULTS_DIR = 'data/docking/ligands','data/docking/receptors','data/docking/results'
for d in [LIGANDS_DIR, RECEPTORS_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)

def pdb_center_of_mass(pdb_path):
    xs, ys, zs = [], [], []
    for line in open(pdb_path):
        if line.startswith(('ATOM','HETATM')):
            try: xs.append(float(line[30:38])); ys.append(float(line[38:46])); zs.append(float(line[46:54]))
            except ValueError: pass
    return (sum(xs)/len(xs), sum(ys)/len(ys), sum(zs)/len(zs)) if xs else (0.0,0.0,0.0)

def to_pdbqt(src, dst):
    r = subprocess.run(['obabel', src, '-O', dst, '--partialcharge','gasteiger'], capture_output=True)
    return os.path.exists(dst) and os.path.getsize(dst) > 100

structures = {}
for cif in glob.glob('data/structures/**/*.cif', recursive=True):
    structures[os.path.basename(os.path.dirname(cif))] = cif
for pdb in glob.glob('data/structures/*.pdb'):
    name = os.path.basename(pdb).replace('.pdb','')
    if 'placeholder' not in name and '_MUT' not in name: structures[name] = pdb

ligands = glob.glob(f'{LIGANDS_DIR}/*.pdbqt')
print(f'Estructuras: {len(structures)} | Ligandos: {len(ligands)}')

if not ligands:
    print('⚠ Sin ligandos PDBQT — omitiendo docking')
else:
    vina_results = {}
    for name, path in structures.items():
        if name.endswith('_WT'): continue
        gene = name.split('_')[0]
        gene_ligs = [l for l in ligands if os.path.basename(l).startswith(gene+'_')]
        if not gene_ligs: continue
        rec_pdbqt = f'{RECEPTORS_DIR}/{name}.pdbqt'
        if not os.path.exists(rec_pdbqt) and not to_pdbqt(path, rec_pdbqt):
            print(f'  ✗ Conversión fallida: {name}'); continue
        tmp = f'/tmp/{name}.pdb'
        if path.endswith('.cif'): subprocess.run(['obabel',path,'-O',tmp], capture_output=True)
        cx, cy, cz = pdb_center_of_mass(tmp if path.endswith('.cif') and os.path.exists(tmp) else path)
        print(f'{name} centro=({cx:.1f},{cy:.1f},{cz:.1f})')
        vina_results[name] = {}
        for lig in gene_ligs:
            drug = os.path.basename(lig).replace('.pdbqt','').replace(f'{gene}_','')
            log  = f'{RESULTS_DIR}/{name}_{drug}_log.txt'
            subprocess.run(['vina','--receptor',rec_pdbqt,'--ligand',lig,
                '--center_x',str(round(cx,2)),'--center_y',str(round(cy,2)),'--center_z',str(round(cz,2)),
                '--size_x','25','--size_y','25','--size_z','25','--exhaustiveness','8',
                '--out',f'{RESULTS_DIR}/{name}_{drug}_out.pdbqt','--log',log], capture_output=True)
            aff = None
            if os.path.exists(log):
                for line in open(log):
                    if line.strip().startswith('1 '):
                        try: aff = float(line.split()[1])
                        except: pass
                        break
            vina_results[name][drug] = aff
            print(f'  {drug}: {aff} kcal/mol')
    with open('data/docking/vina_affinities.json','w') as f: json.dump(vina_results, f, indent=4)
    print('✓ vina_affinities.json guardado')

## M5 — Exportación de resultados para la app web local

**Después de descargar:** descomprime en `D:/ProyectosIA/GEMAMX/data/` y haz `Ctrl+Shift+R`.

In [ ]:
# @title 7. Exportación completa — manifest + metrics + ZIP
import glob, json, os, shutil

# 1. manifest_structures.json
manifest = {}
for cif in glob.glob('data/structures/**/*.cif', recursive=True):
    manifest[os.path.basename(os.path.dirname(cif))] = cif.replace('\\','/')
for pdb in glob.glob('data/structures/*.pdb'):
    name = os.path.basename(pdb).replace('.pdb','')
    if 'placeholder' not in name and '_MUT' not in name:
        manifest[name] = pdb.replace('\\','/')
with open('data/top_variants.json') as f: variants = json.load(f)
for v in variants:
    wt  = f"{v['hugo_symbol']}_WT"
    mut = f"{v['hugo_symbol']}_{v['protein_change']}"
    if wt  not in manifest: manifest[wt]  = None
    if mut not in manifest: manifest[mut] = None
with open('data/manifest_structures.json','w') as f: json.dump(manifest, f, indent=4)
con = sum(1 for v in manifest.values() if v)
print(f'✓ manifest: {len(manifest)} entradas ({con} con archivo, {len(manifest)-con} null)')

# 2. structural_metrics.json
plddt_map = {}
for conf in glob.glob('data/structures/**/confidence_*.json', recursive=True):
    try:
        with open(conf) as f: c = json.load(f)
        name = os.path.basename(conf).replace('confidence_','').replace('_model_0.json','')
        plddt_map[name] = round(c.get('plddt_mean',0)*100, 1)
    except: pass

vina_data = {}
if os.path.exists('data/docking/vina_affinities.json'):
    with open('data/docking/vina_affinities.json') as f: vina_data = json.load(f)

metrics = {}
for v in variants:
    mut_key = f"{v['hugo_symbol']}_{v['protein_change']}"
    entry = {
        'wt_plddt':    plddt_map.get(f"{v['hugo_symbol']}_WT", '--'),
        'mut_plddt':   plddt_map.get(mut_key, '--'),
        'rmsd_global': '--', 'rmsd_active': '--', 'rmsd_regions': []
    }
    if mut_key in vina_data:
        scores = {k: val for k,val in vina_data[mut_key].items() if val is not None}
        if scores:
            best = min(scores, key=scores.get)
            entry['best_vina_drug'] = best
            entry['best_vina_score'] = scores[best]
    metrics[mut_key] = entry
with open('data/structural_metrics.json','w') as f: json.dump(metrics, f, indent=4)
print(f'✓ structural_metrics: {len(metrics)} variantes')

# 3. mutation_stats.json
try:
    gene_dist = df['hugo_symbol'].value_counts().head(20).to_dict()
    with open('data/mutation_stats.json','w') as f:
        json.dump({'gene_distribution': {k:int(v) for k,v in gene_dist.items()},
                   'type_distribution': {}, 'total_samples': int(df['dna_change'].nunique()),
                   'total_mutations': len(df)}, f, indent=4)
    print(f'✓ mutation_stats: {len(gene_dist)} genes, {len(df):,} mutaciones')
except NameError:
    print('⚠ mutation_stats omitido (ejecuta celda 2 primero)')

# 4. ZIP y descarga
shutil.make_archive('gemamx_data','zip','data')
size_mb = os.path.getsize('gemamx_data.zip')/(1024*1024)
print(f'\n✓ gemamx_data.zip ({size_mb:.1f} MB)')
try:
    from google.colab import files
    files.download('gemamx_data.zip')
    print('✅ Descarga iniciada.')
    print('   Descomprime en D:/ProyectosIA/GEMAMX/data/')
    print('   Haz Ctrl+Shift+R en el navegador')
except ImportError:
    print(f'✅ ZIP en: {os.path.abspath("gemamx_data.zip")}')